In [15]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

In [16]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [17]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=instructions,
)

In [18]:
assistant.rag("How do I run Ollama locally?")

ChatCompletion(id='chatcmpl-cd23c9ce-511b-40e2-b08c-f2ead134b628', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To run Ollama locally, follow these steps:\n\n1. **Install Ollama**: \n   - For **macOS**: Download the `.pkg` file from [https://ollama.com/download](https://ollama.com/download) and install it.\n   - For **Windows**: Download the `.msi` file and install it.\n   - For **Linux**: Run the command `curl -fsSL https://ollama.com/install.sh | sh` in your terminal.\n\n2. **Run Ollama**: Open a terminal and type `ollama run llama3`. This will download the LLaMA 3 model (approximately 4GB), start the model locally, and open a chat-like interface where you can type questions.\n\n3. **Test the Ollama local server**: Run `curl http://localhost:11434` in your terminal. You should receive a response similar to `{"models": [...]}`.\n\n4. **Install the Python client**: Install the Ollama Python client using `pip install ollama`.\n\n5.

In [19]:
assistant.rag("How do I run Olama locally?")

ChatCompletion(id='chatcmpl-3fde9206-4df3-41df-b944-c0024fdd5636', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To run Olama (or the course) locally, you will need to set up the following tools:\n\n1. Python\n2. `uv`\n3. Jupyter\n4. Docker\n5. Any other tools needed for the specific module you are working on\n\nHere are the general steps to follow:\n\n1. **Install the required tools**: Make sure you have Python, `uv`, Jupyter, and Docker installed on your machine.\n2. **Set up your environment**: Create a new project directory and navigate to it in your terminal.\n3. **Create a `.env` file**: Add your API keys and other sensitive information to a `.env` file in your project directory. You can use `dirdotenv` to load the environment variables automatically when you `cd` into the project directory.\n4. **Install any additional dependencies**: Depending on the module you are working on, you may need to install additional dependencies

In [20]:
messages = [
    {
        'role': 'system',
        'content': 'You are a course teaching assistant. Use the search tool to answer questions about the LLM Zoomcamp course. Base your answer only on the tool results.'
    },
    {
        'role': 'user',
        'content': 'i just discovered this course, can i join it?'
    }
]

response = groq_client.chat.completions.create(
            messages= messages,
            model="llama-3.3-70b-versatile",
)

In [21]:
response

ChatCompletion(id='chatcmpl-ab248ce5-0e04-4635-b438-dfbf3e084854', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="According to the search results, the LLM Zoomcamp course is a free, online, and open-to-all course. It appears that you can join the course at any time, but it's recommended to start from the beginning to get the most out of it. You can find the course materials, lectures, and assignments on the official website or platform where the course is hosted. However, please note that the course might be self-paced, and some sessions or live meetings might have already taken place. You can still access the recorded sessions and follow along with the course materials.", role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1781800066, model='llama-3.3-70b-versatile', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_32

In [22]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
search('i just discovered this course, can i join it?')

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [24]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

In [25]:
response = groq_client.chat.completions.create(
            messages= messages,
            model="llama-3.3-70b-versatile",
            tools=[search_tool],
            tool_choice="auto",
            temperature=0,
)

In [26]:
assistant_message = response.choices[0].message
tool_call = assistant_message.tool_calls[0]

In [27]:
import json

tool_args = json.loads(tool_call.function.arguments)
search_results = search(tool_args["query"])

In [28]:
assistant_message = response.choices[0].message

messages = [
    messages[0],
    messages[1],
    {
        "role": "assistant",
        "content": assistant_message.content,
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in assistant_message.tool_calls
        ],
    },
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "name": tool_call.function.name,
        "content": json.dumps(search_results),
    },
]

In [29]:
from pprint import pprint
pprint(messages)

[{'content': 'You are a course teaching assistant. Use the search tool to '
             'answer questions about the LLM Zoomcamp course. Base your answer '
             'only on the tool results.',
  'role': 'system'},
 {'content': 'i just discovered this course, can i join it?', 'role': 'user'},
 {'content': None,
  'role': 'assistant',
  'tool_calls': [{'function': {'arguments': '{"query":"joining the LLM '
                                            'Zoomcamp course"}',
                               'name': 'search'},
                  'id': '36vxfvrzj',
                  'type': 'function'}]},
 {'content': '[{"id": "054f3fd58f", "course": "llm-zoomcamp", "section": '
             '"General Course-Related Questions", "question": "Where is the '
             'LLM Zoomcamp Telegram channel?", "answer": "The Telegram channel '
             'is '
             '[https://t.me/llm_zoomcamp](https://t.me/llm_zoomcamp).\\n\\nUse '
             'it for announcements. For technical discussio

In [30]:
tool_functions = {
    "search": search,
}

while True:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=[search_tool],
        tool_choice="auto",
        temperature=0,
    )

    assistant_message = response.choices[0].message

    if not assistant_message.tool_calls:
        print(assistant_message.content)
        break

    messages.append({
        "role": "assistant",
        "content": assistant_message.content,
        "tool_calls": [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments,
                },
            }
            for tc in assistant_message.tool_calls
        ],
    })

    for tool_call in assistant_message.tool_calls:
        tool_args = json.loads(tool_call.function.arguments)
        tool_result = tool_functions[tool_call.function.name](**tool_args)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": tool_call.function.name,
            "content": json.dumps(tool_result),
        })

You don't need a confirmation email. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.
